In [ ]:
%%sql -r dataframe_1
CREATE WAREHOUSE IF NOT EXISTS RETAIL_WH_12
    WITH WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE RETAIL_WH_12;

In [ ]:
%%sql -r dataframe_2
CREATE DATABASE IF NOT EXISTS RETAIL_DW_12;

In [ ]:
%%sql -r dataframe_3
CREATE SCHEMA IF NOT EXISTS RETAIL_DW_12.SALES_ANALYTICS_12;


In [ ]:
%%sql -r dataframe_4
USE DATABASE RETAIL_DW_12;


In [ ]:
%%sql -r dataframe_5
USE SCHEMA SALES_ANALYTICS_12;


In [ ]:
%%sql -r dataframe_6
SELECT CURRENT_DATABASE() AS "Current database", CURRENT_SCHEMA() AS "Current schema";


In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE TABLE DIM_STORE (
    STORE_KEY     NUMBER AUTOINCREMENT START 1 INCREMENT 1 PRIMARY KEY,
    STORE_ID      NUMBER        NOT NULL,
    STORE_NAME    VARCHAR(100),
    CITY          VARCHAR(50),
    STATE         VARCHAR(50),
    STORE_MANAGER VARCHAR(100)          -- SCD Type 1: overwritten in place
);

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE DIM_PRODUCT (
    PRODUCT_KEY   NUMBER AUTOINCREMENT START 1 INCREMENT 1 PRIMARY KEY,
    PRODUCT_ID    NUMBER        NOT NULL,
    PRODUCT_NAME  VARCHAR(100),
    CATEGORY      VARCHAR(50),
    UNIT_PRICE    NUMBER(10,2)
);

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE TABLE DIM_CUSTOMER_HYBRID (
    CUSTOMER_KEY            NUMBER AUTOINCREMENT START 1 INCREMENT 1 PRIMARY KEY,
    CUSTOMER_ID             NUMBER         NOT NULL,
    CUSTOMER_NAME           VARCHAR(100)   NOT NULL,
    CITY                    VARCHAR(50),               -- Type 3, globally synced
    PREVIOUS_CITY           VARCHAR(50),               -- Type 3, globally synced
    STATE                   VARCHAR(50),               -- Type 1, globally synced
    CURRENT_MEMBERSHIP      VARCHAR(30),               -- Type 6, globally synced
    PREVIOUS_MEMBERSHIP     VARCHAR(30),               -- Type 6, globally synced
    HISTORICAL_MEMBERSHIP   VARCHAR(30),               -- Type 6, frozen per row
    SEGMENT                 VARCHAR(30),               -- Type 2, frozen per row
    EFFECTIVE_DATE          DATE           NOT NULL,
    EXPIRY_DATE             DATE           NOT NULL,
    IS_CURRENT              BOOLEAN        NOT NULL DEFAULT TRUE
);

In [ ]:
%%sql -r dataframe_10
CREATE OR REPLACE FILE FORMAT CSV_FORMAT_12
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    NULL_IF = ('NULL', '');

In [ ]:
%%sql -r dataframe_11
CREATE OR REPLACE STAGE RETAIL_STAGE_12
    FILE_FORMAT = CSV_FORMAT_12;

In [ ]:
%%sql -r dataframe_12
LIST @RETAIL_STAGE_12;

In [ ]:
%%sql -r dataframe_13
CREATE OR REPLACE TABLE STG_STORES (
    STORE_ID       NUMBER,
    STORE_NAME     VARCHAR(100),
    CITY           VARCHAR(50),
    STATE          VARCHAR(50),
    STORE_MANAGER  VARCHAR(100)
);

In [ ]:
%%sql -r dataframe_14
COPY INTO STG_STORES
    FROM @RETAIL_STAGE_12/stores.csv
    FILE_FORMAT = (FORMAT_NAME = CSV_FORMAT_12)
    ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
%%sql -r dataframe_15
INSERT INTO DIM_STORE (STORE_ID, STORE_NAME, CITY, STATE, STORE_MANAGER)
SELECT STORE_ID, STORE_NAME, CITY, STATE, STORE_MANAGER
FROM STG_STORES;

In [ ]:
%%sql -r dataframe_16
CREATE OR REPLACE TABLE STG_PRODUCTS (
    PRODUCT_ID    NUMBER,
    PRODUCT_NAME  VARCHAR(100),
    CATEGORY      VARCHAR(50),
    UNIT_PRICE    NUMBER(10,2)
);

In [ ]:
%%sql -r dataframe_17
COPY INTO STG_PRODUCTS
    FROM @RETAIL_STAGE_12/products.csv
    FILE_FORMAT = (FORMAT_NAME = CSV_FORMAT_12)
    ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
%%sql -r dataframe_18
INSERT INTO DIM_PRODUCT (PRODUCT_ID, PRODUCT_NAME, CATEGORY, UNIT_PRICE)
SELECT PRODUCT_ID, PRODUCT_NAME, CATEGORY, UNIT_PRICE
FROM STG_PRODUCTS;

In [ ]:
%%sql -r dataframe_19
CREATE OR REPLACE TABLE STG_CUSTOMERS_INITIAL (
    CUSTOMER_ID    NUMBER,
    CUSTOMER_NAME  VARCHAR(100),
    CITY           VARCHAR(50),
    STATE          VARCHAR(50),
    MEMBERSHIP     VARCHAR(30),
    SEGMENT        VARCHAR(30)
);

In [ ]:
%%sql -r dataframe_20
INSERT INTO DIM_CUSTOMER_HYBRID
    (CUSTOMER_ID, CUSTOMER_NAME, CITY, PREVIOUS_CITY, STATE,
     CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP, HISTORICAL_MEMBERSHIP,
     SEGMENT, EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT)
SELECT
    CUSTOMER_ID, CUSTOMER_NAME, CITY, NULL, STATE,
    MEMBERSHIP, NULL, MEMBERSHIP,
    SEGMENT, '2026-01-01'::DATE, '9999-12-31'::DATE, TRUE
FROM STG_CUSTOMERS_INITIAL;


In [ ]:
%%sql -r dataframe_21
CREATE OR REPLACE TABLE FACT_SALES (
    SALES_KEY        NUMBER AUTOINCREMENT START 1 INCREMENT 1 PRIMARY KEY,
    TRANSACTION_ID   VARCHAR(50)   NOT NULL,
    TRANSACTION_DATE DATE          NOT NULL,
    CUSTOMER_KEY     NUMBER        REFERENCES DIM_CUSTOMER_HYBRID(CUSTOMER_KEY),
    STORE_KEY        NUMBER        REFERENCES DIM_STORE(STORE_KEY),
    PRODUCT_KEY      NUMBER        REFERENCES DIM_PRODUCT(PRODUCT_KEY),
    QUANTITY         NUMBER        NOT NULL,
    UNIT_PRICE       NUMBER(10,2)  NOT NULL,
    TOTAL_AMOUNT     NUMBER(12,2)  NOT NULL
);

In [ ]:
%%sql -r dataframe_22
INSERT INTO FACT_SALES
    (TRANSACTION_ID, TRANSACTION_DATE, CUSTOMER_KEY, STORE_KEY, PRODUCT_KEY,
     QUANTITY, UNIT_PRICE, TOTAL_AMOUNT)
SELECT 'TXN-1001', '2026-02-15'::DATE, c.CUSTOMER_KEY, s.STORE_KEY, p.PRODUCT_KEY,
       1, p.UNIT_PRICE, 1 * p.UNIT_PRICE
FROM DIM_CUSTOMER_HYBRID c, DIM_STORE s, DIM_PRODUCT p
WHERE c.CUSTOMER_ID = 101 AND c.IS_CURRENT = TRUE
  AND s.STORE_ID = 201
  AND p.PRODUCT_ID = 501;

In [ ]:
%%sql -r dataframe_23
INSERT INTO FACT_SALES
    (TRANSACTION_ID, TRANSACTION_DATE, CUSTOMER_KEY, STORE_KEY, PRODUCT_KEY,
     QUANTITY, UNIT_PRICE, TOTAL_AMOUNT)
SELECT 'TXN-1002', '2026-03-10'::DATE, c.CUSTOMER_KEY, s.STORE_KEY, p.PRODUCT_KEY,
       2, p.UNIT_PRICE, 2 * p.UNIT_PRICE
FROM DIM_CUSTOMER_HYBRID c, DIM_STORE s, DIM_PRODUCT p
WHERE c.CUSTOMER_ID = 103 AND c.IS_CURRENT = TRUE
  AND s.STORE_ID = 203
  AND p.PRODUCT_ID = 502;

In [ ]:
%%sql -r dataframe_24


In [ ]:
%%sql -r copy_customers_result
COPY INTO STG_CUSTOMERS_INITIAL
    FROM @RETAIL_STAGE_12/customers_initial.csv
    FILE_FORMAT = (FORMAT_NAME = CSV_FORMAT_12)
    ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
%%sql -r dataframe_25
UPDATE DIM_STORE
SET STORE_MANAGER = 'Suresh Menon'
WHERE STORE_ID = 201;

In [ ]:
%%sql -r dataframe_26
SELECT STORE_ID, STORE_NAME, CITY, STATE, STORE_MANAGER
FROM DIM_STORE
WHERE STORE_ID = 201;

In [ ]:
%%sql -r dataframe_27
CREATE OR REPLACE TABLE STG_CUSTOMER_UPDATES (
    CUSTOMER_ID     NUMBER,
    CUSTOMER_NAME   VARCHAR(100),
    CITY            VARCHAR(50),
    STATE           VARCHAR(50),
    MEMBERSHIP      VARCHAR(30),
    SEGMENT         VARCHAR(30),
    EFFECTIVE_DATE  DATE
);

In [ ]:
%%sql -r dataframe_28
COPY INTO STG_CUSTOMER_UPDATES
    FROM @RETAIL_STAGE_12/customer_updates.csv
    FILE_FORMAT = (FORMAT_NAME = CSV_FORMAT_12)
    ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
%%sql -r dataframe_29
CREATE OR REPLACE TEMPORARY TABLE PRE_UPDATE_SNAPSHOT_12 AS
SELECT d.CUSTOMER_ID, d.CITY AS OLD_CITY, d.CURRENT_MEMBERSHIP AS OLD_MEMBERSHIP
FROM DIM_CUSTOMER_HYBRID d
JOIN STG_CUSTOMER_UPDATES u ON u.CUSTOMER_ID = d.CUSTOMER_ID
WHERE d.IS_CURRENT = TRUE;

In [ ]:
%%sql -r dataframe_30
UPDATE DIM_CUSTOMER_HYBRID t
SET EXPIRY_DATE = DATEADD(day, -1, u.EFFECTIVE_DATE),
    IS_CURRENT  = FALSE
FROM STG_CUSTOMER_UPDATES u
WHERE t.CUSTOMER_ID = u.CUSTOMER_ID
  AND t.IS_CURRENT = TRUE;

In [ ]:
%%sql -r dataframe_31
INSERT INTO DIM_CUSTOMER_HYBRID
    (CUSTOMER_ID, CUSTOMER_NAME, CITY, PREVIOUS_CITY, STATE,
     CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP, HISTORICAL_MEMBERSHIP,
     SEGMENT, EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT)
SELECT
    u.CUSTOMER_ID, u.CUSTOMER_NAME, u.CITY, p.OLD_CITY, u.STATE,
    u.MEMBERSHIP, p.OLD_MEMBERSHIP, u.MEMBERSHIP,
    u.SEGMENT, u.EFFECTIVE_DATE, '9999-12-31'::DATE, TRUE
FROM STG_CUSTOMER_UPDATES u
JOIN PRE_UPDATE_SNAPSHOT_12 p ON p.CUSTOMER_ID = u.CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_32
UPDATE DIM_CUSTOMER_HYBRID t
SET t.CITY                = n.CITY,
    t.PREVIOUS_CITY       = n.PREVIOUS_CITY,
    t.STATE                = n.STATE,
    t.CURRENT_MEMBERSHIP  = n.CURRENT_MEMBERSHIP,
    t.PREVIOUS_MEMBERSHIP = n.PREVIOUS_MEMBERSHIP
FROM (SELECT CUSTOMER_ID, CITY, PREVIOUS_CITY, STATE, CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP
      FROM DIM_CUSTOMER_HYBRID
      WHERE IS_CURRENT = TRUE) n
WHERE t.CUSTOMER_ID = n.CUSTOMER_ID
  AND t.CUSTOMER_ID IN (SELECT CUSTOMER_ID FROM STG_CUSTOMER_UPDATES);


In [ ]:
%%sql -r dataframe_33
INSERT INTO FACT_SALES
    (TRANSACTION_ID, TRANSACTION_DATE, CUSTOMER_KEY, STORE_KEY, PRODUCT_KEY,
     QUANTITY, UNIT_PRICE, TOTAL_AMOUNT)
SELECT 'TXN-2001', '2026-04-15'::DATE, c.CUSTOMER_KEY, s.STORE_KEY, p.PRODUCT_KEY,
       1, p.UNIT_PRICE, 1 * p.UNIT_PRICE
FROM DIM_CUSTOMER_HYBRID c, DIM_STORE s, DIM_PRODUCT p
WHERE c.CUSTOMER_ID = 101 AND c.IS_CURRENT = TRUE   -- resolves to the post-update row
  AND s.STORE_ID = 201
  AND p.PRODUCT_ID = 503;


In [ ]:
%%sql -r dataframe_34
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, PREVIOUS_CITY,
       CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP, HISTORICAL_MEMBERSHIP,
       SEGMENT, EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT
FROM DIM_CUSTOMER_HYBRID
ORDER BY CUSTOMER_ID, EFFECTIVE_DATE;


In [ ]:
%%sql -r dataframe_35
SELECT f.TRANSACTION_ID,
       f.TRANSACTION_DATE,
       c.CUSTOMER_ID,
       c.CUSTOMER_NAME,
       c.CITY               AS CURRENT_CITY,
       c.HISTORICAL_MEMBERSHIP AS MEMBERSHIP_AT_PURCHASE,
       c.SEGMENT            AS SEGMENT_AT_PURCHASE,
       s.STORE_NAME,
       p.PRODUCT_NAME,
       f.TOTAL_AMOUNT
FROM FACT_SALES f
JOIN DIM_CUSTOMER_HYBRID c ON c.CUSTOMER_KEY = f.CUSTOMER_KEY
JOIN DIM_STORE s ON s.STORE_KEY = f.STORE_KEY
JOIN DIM_PRODUCT p ON p.PRODUCT_KEY = f.PRODUCT_KEY
WHERE c.CUSTOMER_ID = 101
ORDER BY f.TRANSACTION_DATE;

In [ ]:
%%sql -r dataframe_37
SELECT 'STORE DIMENSION RECORDS' AS METRIC, COUNT(*) AS VALUE FROM DIM_STORE
UNION ALL
SELECT 'PRODUCT DIMENSION RECORDS', COUNT(*) FROM DIM_PRODUCT
UNION ALL
SELECT 'TOTAL CUSTOMER DIMENSION RECORDS', COUNT(*) FROM DIM_CUSTOMER_HYBRID
UNION ALL
SELECT 'CURRENT CUSTOMER RECORDS', COUNT_IF(IS_CURRENT) FROM DIM_CUSTOMER_HYBRID
UNION ALL
SELECT 'HISTORICAL CUSTOMER RECORDS', COUNT_IF(NOT IS_CURRENT) FROM DIM_CUSTOMER_HYBRID
UNION ALL
SELECT 'FACT SALES TRANSACTIONS', COUNT(*) FROM FACT_SALES;

In [ ]:
%%sql -r dataframe_38
